# G3 multi-doc 질의 검수 집계

동결 대상 `queries.jsonl`에서 무작위로 뽑은 50건의 사람 검수(`data/multidoc/review/g3_sample_review.md`)를 집계한 근거 자료다(PLAN G3). 이 노트북을 보고 세트를 그대로 동결할지, 불량 질의를 뺄지, 규칙을 둘지 정해 D-14에 기록한다.

- 표본은 `rag.multidoc.review`(seed `20260929`)로 뽑았다. 파일럿 통과 질의는 본 생성에서 다시 만들어져 세트에 없어 표본에서 뺐다(사용자)
- 검수 대상은 질의와 정답 문서다. 기대 답·요소·근거는 참고 자료이고, S9 채점은 질의와 정답 문서만 쓴다
- 검색 결과(순위·점수)는 읽지 않는다(SPEC 누수 방지)

In [1]:
import hashlib
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf
from scipy.stats import binomtest

from rag.index import load_cfg
from rag.multidoc.review import doc_heads, scope_target
from rag.sample_review import sample_qids

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg()
QUERIES = ROOT / cfg.paths.multidoc_queries
CHECK_IN = ROOT / cfg.paths.multidoc_check_in
REVIEW = ROOT / "data/multidoc/review/g3_sample_review.md"
SCOPE_REVIEW = ROOT / "data/multidoc/review/g3_scope_review.md"
EXCLUDE = ROOT / cfg.paths.multidoc_exclude
FROZEN = ROOT / cfg.paths.multidoc_frozen
JUDGE_OUT = ROOT / cfg.paths.multidoc_judge_out

In [3]:
def parse_review(text):
    # 건마다 '## qid' 절의 검수 칸(- 판정: 등)을 읽는다
    rows = []
    for sec in re.split(r"^## ", text, flags=re.M)[1:]:
        qid = sec.split("\n", 1)[0].strip()
        box = sec.split("**검수**", 1)[1]
        pairs = re.findall(r"^- (판정|불량 유형|정답 문서|메모):(.*)$", box, re.M)
        field = {k: v.strip() for k, v in pairs}
        rows.append({"qid": qid, **field})
    return pd.DataFrame(rows)


def pool_docs(pool_id):
    # 검사 입력의 문서 번호별 본문(머리 줄 포함)
    body = (CHECK_IN / f"{pool_id}.md").read_text(encoding="utf-8").split("문서:", 1)[1]
    parts = re.split(r"^\[(\d+)\] ", body, flags=re.M)
    return {int(parts[i]): parts[i + 1] for i in range(1, len(parts), 2)}

## 로딩

In [4]:
queries = {q["qid"]: q for q in map(json.loads, QUERIES.open(encoding="utf-8"))}
print(len(queries), hashlib.sha256(QUERIES.read_bytes()).hexdigest())
print(Counter(q["type"] for q in queries.values()))

638 d2169035b3c5940069fd6a8b62e1f43820d6c50c1234b96a4a92201a8cc48061
Counter({'conf': 233, 'questioner': 215, 'law': 190})


In [5]:
review = parse_review(REVIEW.read_text(encoding="utf-8"))
review["type"] = review["qid"].map(lambda k: queries[k]["type"])
picked = sample_qids(list(queries), cfg.seed, cfg.g3_sample_size)
print(review.shape, sorted(review["qid"]) == picked)
review.head()

(50, 6) True


,qid,판정,불량 유형,정답 문서,메모,type
0,md-conf-0017,통과,,,"하위 질문이 D1(영향평가 지연)과 D7(회수, 사후관리)로 나뉨.",conf
1,md-conf-0032,통과,,,"D1 처리기간·연장, D4 직권취소, D13 도입 취지로 하위 질문마다 문서가 대응함.",conf
2,md-conf-0077,통과,,,"네 수단이 D2(가산금), D10(등록 제한), D14(명예감시원), D4(사전통보...",conf
3,md-conf-0090,통과,,,요소 5(김정재 질의)는 질의만 있고 정부 답변(특별한 혜택 없음) 근거가 없음. ...,conf
4,md-conf-0105,통과,,,"D5 현황, D13 지방분권 논의 방향.",conf


## 판정 집계

구간은 Clopper-Pearson 95%다. 표본이 50건이라 구간이 넓다.

In [6]:
def rate(g):
    k, n = (g["판정"] == "불량").sum(), len(g)
    ci = binomtest(k, n).proportion_ci(method="exact")
    return pd.Series(
        {"표본": n, "불량": k, "불량률": k / n, "95% 하한": ci.low, "95% 상한": ci.high}
    )


summary = pd.concat([review.groupby("type").apply(rate), rate(review).to_frame("전체").T])
summary.round(3)

,표본,불량,불량률,95% 하한,95% 상한
conf,14.0,0.0,0.000,0.000,0.232
law,11.0,3.0,0.273,0.060,0.610
questioner,25.0,0.0,0.000,0.000,0.137
전체,50.0,3.0,0.060,0.013,0.165


In [7]:
review[review["판정"] == "불량"][["qid", "불량 유형", "정답 문서", "메모"]]

,qid,불량 유형,정답 문서,메모
14,md-law-0012,질의 모호,,질의가 주제 없이 두 회의의 '질의와 답변의 내용' 전체를 물음. 추경 심사 회의는...
19,md-law-0108,질의 모호,,특위가 여러 성폭력 법안을 함께 심사해 회의마다 쟁점이 여럿임(기대 답 기준 11월...
22,md-law-0190,기타,원문 확인 필요(후보 D2),질의 전제와 정답 내용 불일치. 질의는 학술진흥법·학자금대출 신용보증법안 심사에서 ...


## 불량 건의 정답과 평가 용도

S9 채점은 질의와 정답 문서만 쓴다. 불량 3건이 검색 평가에도 문제가 되는지는 정답 문서가 질의가 지목한 회의의 문서를 빠짐없이 담는지로 본다. 질의가 회의 날짜로 대상을 정하므로, 그 회의의 코퍼스 문서가 정답에 모두 있으면 질의가 모호해도 검색 정답은 맞다.

In [8]:
defects = review.loc[review["판정"] == "불량", "qid"].tolist()
meeting_docs = Counter()
gold_meetings = {d.split("-")[0] for k in defects for d in queries[k]["gold_doc_ids"]}
with (ROOT / cfg.paths.corpus).open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["conference_number"] in gold_meetings:
            meeting_docs[d["conference_number"]] += 1

for k in defects:
    q = queries[k]
    heads = doc_heads((CHECK_IN / f"{q['pool_id']}.md").read_text(encoding="utf-8"))
    print(k, q["query"])
    for n, d in enumerate(q["pool_doc_ids"], 1):
        mark = "정답" if d in q["gold_doc_ids"] else "    "
        same = meeting_docs.get(d.split("-")[0], "-")
        print(f"  {mark} D{n} {heads[n][:45]} | 코퍼스 같은 회의 문서 {same}")

md-law-0012 2002년도 제1회 추가경정예산안 심사에서 예산결산특별위원회(제234회)가 2002년 9월 12일 제1차 회의와 9월 13일 제2차 회의에서 각각 다룬 질의와 답변의 내용은 무엇인가?
  정답 D1 2002年9月12日(木) 예산결산특별위원회 제234회 제1차 | 코퍼스 같은 회의 문서 1
  정답 D2 2002年9月13日(金) 예산결산특별위원회 제234회 제2차 | 코퍼스 같은 회의 문서 1
md-law-0108 아동ㆍ여성대상성폭력대책특별위원회 법안심사소위원회의 2012년 10월 17일, 11월 2일, 11월 5일 회의에서 성폭력 관련 법안 심사 중 각각 논의된 쟁점은 무엇인가?
       D1 2012년10월4일(목) 아동ㆍ여성대상성폭력대책특별위원회 제311회 제3차 | 코퍼스 같은 회의 문서 -
  정답 D2 2012년10월17일(수) 아동.여성대상성폭력대책특별위원회-법안심사소위원회 제31 | 코퍼스 같은 회의 문서 1
  정답 D3 2012년11월5일(월) 아동.여성대상성폭력대책특별위원회-법안심사소위원회 제311 | 코퍼스 같은 회의 문서 1
  정답 D4 2012년11월2일(금) 아동.여성대상성폭력대책특별위원회-법안심사소위원회 제311 | 코퍼스 같은 회의 문서 1
md-law-0190 교육위원회 법안심사소위원회가 학술진흥법 일부개정법률안과 학자금대출 신용보증법안을 계속 심사한 2005년 5월 2일 회의(제253회 제5차)와 2005년 6월 15일 회의(제254회 제2차)에서 각각 위원들이 제기한 쟁점은?
  정답 D1 2005年5月2日(月) 교육위원회-법안심사소위원회 제253회 제5차 | 코퍼스 같은 회의 문서 1
       D2 2005年6月3日(金) 교육위원회-법안심사소위원회 제254회 제1차 | 코퍼스 같은 회의 문서 -
  정답 D3 2005年6月15日(水) 교육위원회-법안심사소위원회 제254회 제2차 | 코퍼스 같은 회의 문서 1


세 건 모두 질의가 지목한 회의의 문서가 코퍼스에 회의당 하나이고, 그 문서가 모두 정답에 있다. `md-law-0108`의 D1(10월 4일 전체회의)은 질의가 지목하지 않은 회의다. 아래는 검수 메모에서 원문 확인이 필요하다고 남긴 내용이다.

In [9]:
def hits(pool_id, words):
    for n, text in pool_docs(pool_id).items():
        print(f"D{n}", {w: len(re.findall(w, text)) for w in words}, f"본문 {len(text)}자")


print("law-0190: 두 법안 이름 등장 횟수(안건 머리 1회 포함)")
hits("law-0190", ["학술진흥", "학자금"])
print("law-0108: 기대 답의 신상정보 고지·신상공개 일원화")
hits("law-0108", ["고지", "신상공개", "일원화"])

law-0190: 두 법안 이름 등장 횟수(안건 머리 1회 포함)
D1 {'학술진흥': 1, '학자금': 1} 본문 1556자
D2 {'학술진흥': 1, '학자금': 6} 본문 8183자
D3 {'학술진흥': 1, '학자금': 1} 본문 2059자
law-0108: 기대 답의 신상정보 고지·신상공개 일원화
D1 {'고지': 0, '신상공개': 0, '일원화': 6} 본문 7257자
D2 {'고지': 0, '신상공개': 0, '일원화': 0} 본문 3840자
D3 {'고지': 3, '신상공개': 3, '일원화': 2} 본문 13127자
D4 {'고지': 6, '신상공개': 0, '일원화': 0} 본문 5986자


- `law-0190`: 6월 15일 회의(D3)는 안건에 두 법안이 있지만 본문에는 두 법안 이름이 나오지 않는다. 두 법안 논의는 질의가 지목하지 않은 6월 3일 회의(D2)에 있다. 질의 전제(두 법안 심사에서 나온 쟁점)로 검색하면 내용상 맞는 문서는 D2인데 정답은 D3이다. 세 건 가운데 검색 정답이 질의 내용과 어긋나는 건은 이것뿐이다
- `law-0108`: 신상정보 고지는 11월 2일 회의(D4), 신상공개 일원화는 11월 5일 회의(D3)에 있다. 정답 D2·D3·D4가 맞다
- `law-0012`: 후보 집합이 두 회의 문서 두 개뿐이고 둘 다 정답이다

In [10]:
q = queries["md-law-0202"]
for el in q["elements"]:
    quotes = [s["quote"] for s in el["support"]]
    if any(w in el["fact"] for w in ["11만", "24만"]) or any("24만" in t for t in quotes):
        print(el["fact"])
        print(" ", [t[:120] for t in quotes])
text = pool_docs(q["pool_id"])[3]
i = text.find("11만 명")
print(text[i - 60 : i + 90])

2018년 8월 27일: 공급이 확대되면 기존 우유급식 체계 등을 활용 가능하다는 답변과, 본사업 확대 시 공급체계가 달라져야 한다는 논의
  ['채 위원님이 말씀하신 대로 앞으로 공급이 확대되면 기존의 우유급식 체계나 그런 것들을 충분히 활용 가능하다 저희도 그렇게 생각하고 있습니다.', '24만 명 대상으로 한 시범사업은 지금 저희들이 하고 있는 체제로 가능합니다만 그게 만약에 본사업으로 해서 초등학교 전체를 한다든지 이렇게 되면 이 공급체계가 훨씬 달라져야 된다고 생각합니다.']
데…… 지금은 시범사업이니까 10개가 가능하겠지요. 지금 시범사업에서 하고 있는 것이 10개를 하고 있는데 11만 명 정도를 공급하고 있습니다.  그런데 돌봄교실이 전체가 한 24만 명 정도 되는데 만약에 기존의 급식체계, 급식을 하는 업체가 하는 것이 효율적이라 그러


`law-0202`(통과)의 수치 차이: 원문에 '시범사업에서 11만 명 정도를 공급'과 '24만 명 대상으로 한 시범사업'(요소 근거)이 함께 있다. 발언 안에서 두 수치가 섞여 있어 기대 답이나 요소 어느 한쪽의 오류로 보기 어렵다.

## law 질의 형태

불량 3건은 모두 law다. 사용자 검수 기준(2026-10-08): '각 회의에서 논의된 쟁점은?' 형식은 열거형으로 볼 측면이 있지만 답의 범위가 정해지지 않으면 `질의 모호`다. 표본에서는 같은 형식이라도 법안 하나로 범위가 좁혀진 질의(`law-0079`, `law-0112`)는 통과했고, 회의 주제가 여럿인 추경 심사(`law-0012`)와 여러 법안을 함께 심사한 특위(`law-0108`)는 불량이었다. 아래는 '각각'과 '쟁점' 또는 '질의와 답변'이 함께 나오는 질의를 문자열로 센 것이다. 범위가 좁혀졌는지는 문자열로 가를 수 없어 상한으로만 본다.

In [11]:
def per_meeting(text):
    return "각각" in text and bool(re.search(r"쟁점|질의와 답변", text))


full = pd.DataFrame(queries.values())
full["회의별 쟁점형"] = full["query"].map(per_meeting)
print(pd.crosstab(full["type"], full["회의별 쟁점형"]))
law = review[review["type"] == "law"].copy()
law["회의별 쟁점형"] = law["qid"].map(lambda k: per_meeting(queries[k]["query"]))
pd.crosstab(law["회의별 쟁점형"], law["판정"])

회의별 쟁점형     False  True 
type                    
conf          230      3
law            70    120
questioner    214      1


판정,불량,통과
회의별 쟁점형,,
False,0,4
True,3,4


## 통과 건 메모 묶음

통과 47건 중 메모가 있는 건을 메인이 유형으로 묶었다. 모두 기대 답·요소 쪽 문제이고, S9 채점(질의·정답 문서)에는 쓰이지 않는다.

In [12]:
memo_group = {
    # 기대 답 내용에 요소 근거 없음, 범위 다름, 근거가 요소를 뒷받침하지 않음
    "기대 답·요소 불일치": [
        "md-conf-0090", "md-conf-0188", "md-conf-0197", "md-conf-0216", "md-conf-0293",
        "md-conf-0313", "md-law-0035", "md-law-0193", "md-law-0202", "md-questioner-0135",
        "md-questioner-0164", "md-questioner-0293",
    ],
    "요소에 진행 발언·주장 포함": [
        "md-questioner-0035", "md-questioner-0043", "md-questioner-0106",
    ],
    "단일 문서 경계(통과 판정)": ["md-conf-0199"],
}
passed = set(review.loc[review["판정"] == "통과", "qid"])
assert all(k in passed for ks in memo_group.values() for k in ks)
pd.Series({g: len(ks) for g, ks in memo_group.items()}, name="건수")

기대 답·요소 불일치        12
요소에 진행 발언·주장 포함     3
단일 문서 경계(통과 판정)     1
Name: 건수, dtype: int64

## law 범위 재판정

사용자 결정(2026-10-08): 위 형식의 law 질의 중 표본 밖 113건을 사용자가 직접 재판정하고(`rag.multidoc.review --scope`, 질의와 정답 회의의 안건만 보임), `모호`는 세트에서 뺀다. 판정 기준은 표본 검수와 같다. 법안 하나나 주제로 범위가 정해지면 `통과`, 회의 주제가 여럿인데 범위를 정하지 않으면 `모호`다.

In [13]:
def scope_row(sec):
    qid = sec.split("\n", 1)[0].strip()
    return {"qid": qid, "판정": sec.split("- 판정:", 1)[1].split("\n", 1)[0].strip()}


sections = re.split(r"^## ", SCOPE_REVIEW.read_text(encoding="utf-8"), flags=re.M)[1:]
scope = pd.DataFrame([scope_row(sec) for sec in sections])
s = cfg.g3_scope
target = sorted(
    k for k, q in queries.items()
    if q["type"] == s.type and scope_target(q["query"], s.must, s.any) and k not in set(picked)
)
print(scope.shape, sorted(scope["qid"]) == target)
scope["판정"].value_counts()

(113, 2) True


판정
통과    93
모호    20
Name: count, dtype: int64

In [14]:
pattern_law = pd.concat([
    law.loc[law["회의별 쟁점형"], ["qid", "판정"]].assign(출처="표본"),
    scope.assign(출처="재판정"),
], ignore_index=True)
pd.crosstab(pattern_law["출처"], pattern_law["판정"], margins=True)

판정,모호,불량,통과,All
출처,,,,
재판정,20,0,93,113
표본,0,3,4,7
All,20,3,97,120


## law 재판정 (서브에이전트)

범위 재판정 파일은 질의와 안건만 보여서 정답 문서 내용 불일치(`law-0190` 유형)는 판정할 수 없었다(사용자 지적, 2026-10-08). 사용자 결정으로 law 질의 190건(동결분 167건과 이미 뺀 23건)을 Sonnet 서브에이전트가 정답 문서 본문까지 읽고 `통과`·`모호`·`불일치`로 다시 판정했다(`rag.multidoc.review --judge`, 지시 `scope_v2`). 이미 뺀 23건은 사람 판정과 대조하려고 넣었다. 처음 지시 `scope_v1`은 입력에 질의자가 없어 위원을 지목한 질의의 화자를 확인할 수 없었고, 106건을 판정한 뒤 질의자를 넣은 `scope_v2`로 전체를 다시 판정했다. `scope_v1` 출력은 `judge_out/scope_v1/`에 남겨 실행 간 일치를 본다.

In [15]:
def load_judge(d):
    return {f.stem: json.loads(f.read_text(encoding="utf-8")) for f in sorted(d.glob("*.json"))}


v2, v1 = load_judge(JUDGE_OUT), load_judge(JUDGE_OUT / "scope_v1")
law_qids = sorted(k for k, q in queries.items() if q["type"] == "law")
print(len(v2), sorted(v2) == law_qids, len(v1))
pd.Series(Counter(d["verdict"] for d in v2.values()))

190 True 106


통과     157
모호      25
불일치      8
dtype: int64

In [16]:
human = {
    **dict(zip(review["qid"], review["판정"], strict=True)),
    **dict(zip(scope["qid"], scope["판정"], strict=True)),
}
cmp = pd.DataFrame(
    [{"qid": k, "사람": human[k], "작업자": v2[k]["verdict"]} for k in law_qids if k in human]
)
cmp["일치"] = (cmp["사람"] == "통과") == (cmp["작업자"] == "통과")
print(len(cmp), cmp["일치"].sum())
pd.crosstab(cmp["사람"], cmp["작업자"])

124 116


작업자,모호,불일치,통과
사람,,,
모호,17,1,2
불량,2,1,0
통과,2,4,95


In [17]:
both = sorted(set(v1) & set(v2))
flip = [
    (k, v1[k]["verdict"], v2[k]["verdict"])
    for k in both
    if v1[k]["verdict"] != v2[k]["verdict"]
]
print(len(both), len(both) - len(flip))
pd.DataFrame(flip, columns=["qid", "scope_v1", "scope_v2"])

106 97


,qid,scope_v1,scope_v2
0,md-law-0001,통과,모호
1,md-law-0020,불일치,통과
2,md-law-0024,모호,통과
3,md-law-0083,모호,통과
4,md-law-0096,통과,모호
5,md-law-0098,통과,불일치
6,md-law-0103,통과,모호
7,md-law-0122,불일치,통과
8,md-law-0134,통과,불일치


`law-0122`는 질의자를 넣어 바뀐 것이고(v1 불일치: 화자 확인 불가 → v2 통과), 나머지는 같은 입력에서도 판정이 갈린 것이다. 사람 판정과 통과 여부가 맞은 비율은 위 표와 같다.

In [18]:
excluded_before = set(review.loc[review["판정"] == "불량", "qid"]) | set(
    scope.loc[scope["판정"] == "모호", "qid"]
)
flagged = sorted(
    k for k in law_qids if k not in excluded_before and v2[k]["verdict"] != "통과"
)
pd.DataFrame(
    [{"qid": k, "판정": v2[k]["verdict"], "사람": human.get(k, "-"), "이유": v2[k]["reason"][:80]}
     for k in flagged]
)

,qid,판정,사람,이유
0,md-law-0001,모호,-,"두 회의 모두 국가공무원법 일부개정법률안이 여러 건(1차 4건, 2차 3건) 안건이..."
1,md-law-0006,불일치,-,문서 1과 2는 질의의 법안 논의가 본문에 있다. 문서 3은 안건에는 질의의 법안이...
2,md-law-0025,모호,통과,"2013년 회의는 정부 제출 형법 개정안 하나라 범위가 정해지지만, 2015년 12..."
3,md-law-0028,모호,-,두 회의 모두 소득세법·법인세법·조세특례제한법 등 50여 건의 세법 개정안과 청원을...
4,md-law-0031,불일치,통과,3번 문서(2007년 9월 10일)는 임시직(일용직) 보수와 신분보장을 묻고 답하는...
5,md-law-0098,불일치,통과,문서 2는 취업 후 학자금 상환제도의 재원(국채 대 장학재단 채권)을 논의해 질의에...
6,md-law-0115,불일치,통과,"문서 1은 영해 및 접속수역법 개정안의 벌금액 상향 논의를 담고 있으나, 문서 2는..."
7,md-law-0127,모호,-,두 회의가 24개 법안을 함께 다루는데 질의가 법안이나 주제를 정하지 않아 무엇을 ...
8,md-law-0134,불일치,-,"문서 1은 임종인 위원이 군인사법 개정안(군법무관 장기복무, 시보 기간 가산)을 논..."
9,md-law-0159,모호,통과,"세 문서 모두 본문에 부가가치세법 개정안 논의가 있으나, 11월 30일 회의는 6건..."


사용자 결정(2026-10-08): 동결분 law 중 작업자가 통과가 아니라고 판정한 위 12건을 모두 뺀다. 사람이 통과로 본 질의(`law-0025`, `law-0159` 모호)도 포함한다. 실행마다 갈리는 판정이 있어, 이번 실행에서 통과였던 질의 중에도 다시 판정하면 걸리는 것이 있을 수 있다.

## 동결 세트

제외 목록은 `configs/multidoc/g3_exclude.yaml`에 두고 `rag.multidoc.freeze`가 `queries.jsonl`에서 빼서 `queries_g3.jsonl`을 쓴다. `queries.jsonl`은 검사 기록에서 다시 만들어지는 파일이라 고치지 않는다.

In [19]:
exclude = OmegaConf.to_container(OmegaConf.load(EXCLUDE))
drop = {k for ks in exclude.values() for k in ks}
assert set(exclude["sample"]) == set(review.loc[review["판정"] == "불량", "qid"])
assert set(exclude["scope"]) == set(scope.loc[scope["판정"] == "모호", "qid"])
assert set(exclude["judge"]) == set(flagged)
frozen = [json.loads(line) for line in FROZEN.open(encoding="utf-8")]
print({k: len(v) for k, v in exclude.items()}, len(frozen))
print([q["qid"] for q in frozen] == [k for k in queries if k not in drop])
print(hashlib.sha256(FROZEN.read_bytes()).hexdigest())

{'sample': 3, 'scope': 20, 'judge': 12} 603
True
ad8a8fc3aa0192c805f55d00b2eac8a097be71002dca0c173492534e436f8cf4


In [20]:
pd.DataFrame({
    "검수 전": Counter(q["type"] for q in queries.values()),
    "제외": Counter(queries[k]["type"] for k in drop),
    "동결": Counter(q["type"] for q in frozen),
}).fillna(0).astype(int)

,검수 전,제외,동결
conf,233,0,233
law,190,35,155
questioner,215,0,215
